[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/claude-code/04-flujos-desarrollo-ia.ipynb)

# Flujos de desarrollo con IA — Notebook interactivo

Implementamos los flujos de trabajo más útiles de Claude Code:
exploración de código, debug, revisión de PR y generación de documentación.

In [ ]:
import anthropic
import json
from datetime import datetime

client = anthropic.Anthropic()
print("Cliente listo")

## 1. Explorar y entender código

El flujo más frecuente: entender qué hace un fragmento de código desconocido.

In [ ]:
CODIGO_LEGACY = '''
def proc_data(d, f=None, r=False):
    res = []
    for i, x in enumerate(d):
        if f and not f(x): continue
        t = x.get('val', 0) * 1.21 if x.get('iva') else x.get('val', 0)
        if r: t = -t
        res.append({'id': x.get('id', i), 'total': round(t, 2),
                    'ts': x.get('ts', ''), 'ok': t > 0})
    return sorted(res, key=lambda x: x['total'], reverse=True)
'''

def explorar_codigo(codigo: str) -> dict:
    """Equivalente a preguntar a Claude Code sobre un archivo."""
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=600,
        system="Eres un senior engineer revisando código legacy. Sé conciso y práctico.",
        messages=[{"role": "user", "content": f"""Analiza este código y responde en JSON:
{{
  "que_hace": "descripción clara en 1-2 frases",
  "parametros": {{"nombre_real": "descripción"}},
  "problemas": ["lista de problemas de legibilidad/mantenimiento"],
  "version_refactorizada_nombre": "nombre sugerido para la función",
  "prioridad_refactor": "alta|media|baja"
}}

CÓDIGO:
{codigo}"""}]
    )
    return json.loads(resp.content[0].text)

analisis = explorar_codigo(CODIGO_LEGACY)
print("ANÁLISIS DEL CÓDIGO:")
print(json.dumps(analisis, indent=2, ensure_ascii=False))

In [ ]:
def refactorizar_codigo(codigo: str, contexto: str = "") -> str:
    """Equivalente a pedirle a Claude Code que refactorice un fragmento."""
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=800,
        system="Refactoriza manteniendo la funcionalidad exacta. Usa type hints de Python 3.11.",
        messages=[{"role": "user", "content": f"""Refactoriza este código:
- Nombres descriptivos (sin abreviaciones)
- Type hints completos
- Docstring conciso
- Sin cambiar la lógica de negocio
{f'Contexto: {contexto}' if contexto else ''}

CÓDIGO ORIGINAL:
{codigo}

Solo el código refactorizado, sin explicaciones."""}]
    )
    return resp.content[0].text

refactorizado = refactorizar_codigo(
    CODIGO_LEGACY,
    contexto="Es una función de procesamiento de facturas con IVA. 'r' indica abonos/devoluciones."
)
print("CÓDIGO REFACTORIZADO:")
print(refactorizado)

## 2. Debug de errores

In [ ]:
ERROR_PRODUCCION = """
Traceback (most recent call last):
  File "/app/src/services/invoice_service.py", line 87, in process_batch
    total = sum(item['amount'] for item in items)
  File "/app/src/services/invoice_service.py", line 87, in <genexpr>
    total = sum(item['amount'] for item in items)
KeyError: 'amount'

Durante el procesamiento del batch #4821 (142 facturas)
Empezó a las 14:32 UTC tras desplegar v2.4.1
"""

CODIGO_RELACIONADO = """
# invoice_service.py:80-95
def process_batch(invoice_ids: list[str]) -> dict:
    items = db.query(Invoice).filter(Invoice.id.in_(invoice_ids)).all()
    items_dict = [{
        'id': item.id,
        'importe': item.total_amount,  # <- campo renombrado en v2.4.1
        'fecha': item.created_at
    } for item in items]
    
    total = sum(item['amount'] for item in items_dict)  # <- usa nombre viejo
    return {'total': total, 'count': len(items_dict)}
"""

def diagnosticar_error(error: str, codigo: str) -> dict:
    """Equivalente al flujo de debug con Claude Code."""
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=600,
        messages=[{"role": "user", "content": f"""Analiza este error de producción. JSON:
{{
  "causa_raiz": "explicación en 1 frase",
  "linea_problema": "línea exacta con el bug",
  "fix": "código corregido (solo la línea)",
  "test_regresion": "test que evitaría que vuelva a ocurrir",
  "riesgo_datos": "¿se corrompieron datos? ¿qué verificar?"
}}

ERROR:
{error}

CÓDIGO:
{codigo}"""}]
    )
    return json.loads(resp.content[0].text)

diagnostico = diagnosticar_error(ERROR_PRODUCCION, CODIGO_RELACIONADO)
print("DIAGNÓSTICO:")
print(json.dumps(diagnostico, indent=2, ensure_ascii=False))

## 3. Code Review como senior engineer

In [ ]:
CODIGO_A_REVISAR = '''
from fastapi import FastAPI
import sqlite3

app = FastAPI()
DB_PATH = "users.db"

@app.get("/users/{user_id}")
def get_user(user_id: str):
    con = sqlite3.connect(DB_PATH)
    cursor = con.cursor()
    query = f"SELECT * FROM users WHERE id = {user_id}"
    cursor.execute(query)
    user = cursor.fetchone()
    con.close()
    return {"user": user}

@app.post("/login")
def login(username: str, password: str):
    con = sqlite3.connect(DB_PATH)
    cursor = con.cursor()
    result = cursor.execute(
        f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
    ).fetchone()
    return {"logged_in": result is not None, "user": result}
'''

def revisar_codigo(codigo: str) -> dict:
    """Equivalente al comando /review de Claude Code."""
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=800,
        system="Eres un senior security engineer revisando código antes del merge. Sé crítico y específico.",
        messages=[{"role": "user", "content": f"""Revisa este código. JSON:
{{
  "vulnerabilidades": [
    {{"tipo": "nombre", "linea": 0, "descripcion": "...", "severidad": "critica|alta|media|baja", "fix": "código corregido"}}
  ],
  "problemas_calidad": ["lista"],
  "aprobado": false,
  "puntuacion_seguridad": 0,
  "resumen": "1 frase"
}}

CÓDIGO:
{codigo}"""}]
    )
    return json.loads(resp.content[0].text)

review = revisar_codigo(CODIGO_A_REVISAR)
print(f"RESULTADO DEL REVIEW:")
print(f"Aprobado: {'✅' if review['aprobado'] else '❌'}")
print(f"Puntuación de seguridad: {review['puntuacion_seguridad']}/10")
print(f"Resumen: {review['resumen']}")
print(f"\nVulnerabilidades encontradas: {len(review['vulnerabilidades'])}")
for v in review['vulnerabilidades']:
    icono = {"critica": "🔴", "alta": "🟠", "media": "🟡", "baja": "🟢"}.get(v['severidad'], "⚪")
    print(f"  {icono} [{v['severidad'].upper()}] Línea {v['linea']}: {v['tipo']}")
    print(f"     {v['descripcion']}")

## 4. Generación de tests automática

In [ ]:
FUNCION_A_TESTEAR = '''
def calcular_descuento(precio: float, cliente_tipo: str, cantidad: int) -> float:
    """Calcula el precio final con descuentos aplicados."""
    if precio < 0:
        raise ValueError("El precio no puede ser negativo")
    
    descuento = 0.0
    
    # Descuento por tipo de cliente
    if cliente_tipo == "premium":
        descuento += 0.15
    elif cliente_tipo == "enterprise":
        descuento += 0.25
    
    # Descuento por volumen
    if cantidad >= 100:
        descuento += 0.10
    elif cantidad >= 10:
        descuento += 0.05
    
    # Límite máximo de descuento: 30%
    descuento = min(descuento, 0.30)
    
    return round(precio * (1 - descuento), 2)
'''

def generar_tests(codigo: str) -> str:
    """Genera tests pytest para una función."""
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=800,
        system="Genera tests pytest exhaustivos. Incluye casos normales, borde y de error.",
        messages=[{"role": "user", "content": f"""Genera tests pytest para esta función.
Incluye: casos normales, casos borde, casos de error.
Usa nombres descriptivos. Solo el código, sin explicaciones.

FUNCIÓN:
{codigo}"""}]
    )
    return resp.content[0].text

tests = generar_tests(FUNCION_A_TESTEAR)
print("TESTS GENERADOS:")
print(tests)

## 5. Comparativa de herramientas IA para desarrollo

In [ ]:
COMPARATIVA = {
    "Tarea": [
        "Autocompletado inline mientras escribes",
        "Entender un repo nuevo completo",
        "Refactorización de múltiples archivos",
        "Debug con contexto de toda la app",
        "Ejecutar tests y corregir fallos",
        "Conectar con tu BD o API interna",
        "Revisar PR con contexto completo",
        "Automatizar tareas repetitivas (hooks)",
        "Uso sin cambiar de IDE",
    ],
    "Claude Code": ["⭐", "⭐⭐⭐", "⭐⭐⭐", "⭐⭐⭐", "⭐⭐⭐", "⭐⭐⭐ (MCP)", "⭐⭐⭐", "⭐⭐⭐ (Hooks)", "⭐⭐ (Terminal)"],
    "Cursor": ["⭐⭐⭐", "⭐⭐", "⭐⭐", "⭐⭐", "⭐⭐", "⭐ (experimental)", "⭐⭐", "⭐", "⭐⭐⭐ (propio IDE)"],
    "Copilot": ["⭐⭐⭐", "⭐", "⭐", "⭐", "⭐", "❌", "⭐", "❌", "⭐⭐⭐ (plugin)"],
}

# Imprimir tabla
col_w = 35
print(f"{'Tarea':<{col_w}} {'Claude Code':^15} {'Cursor':^10} {'Copilot':^10}")
print("-" * 75)
for i, tarea in enumerate(COMPARATIVA["Tarea"]):
    print(f"{tarea:<{col_w}} {COMPARATIVA['Claude Code'][i]:^15} {COMPARATIVA['Cursor'][i]:^10} {COMPARATIVA['Copilot'][i]:^10}")

print("\n⭐⭐⭐ = Excelente | ⭐⭐ = Bueno | ⭐ = Básico | ❌ = No soportado")
print("\n💡 Recomendación: Claude Code + Copilot es el stack más completo.")
print("   Copilot para autocompletado inline, Claude Code para tareas complejas.")